# Basic Multi‑Agent Reference (LangGraph + LangChain + OpenAI)

This notebook is a minimal reference implementation of a **multi‑agent** workflow using **LangGraph** (orchestrator) and **LangChain** (LLM wrapper) with **OpenAI** as the model provider.

**What you can change easily:**
- `MODEL` (via `OPENAI_MODEL`)
- agent prompts (`PLANNER_PROMPT`, `EXECUTOR_PROMPT`)
- the task you send in (`task`)

> Requires: an OpenAI API key in `OPENAI_API_KEY` (environment variable or `.env`).

In [ ]:
# If needed (first time only), install deps:
# %pip install -U langgraph langchain langchain-openai openai python-dotenv

import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1")
print("Using model:", MODEL)


## 1) Building block: LLM interface (LangChain `ChatOpenAI`)

This is the smallest useful wrapper: call the model with a list of chat messages.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

msg = llm.invoke([
    ("system", "You are a concise assistant."),
    ("user", "Say hello in one sentence and ask what problem I'm solving."),
])

msg.content


## 2) Prompts you will tune

These prompts are the main knobs participants typically tweak during a workshop.

In [ ]:
PLANNER_PROMPT = """
You are a planning agent.
Given the user's task, create a short, actionable plan with 3-7 steps.
Keep it tool-agnostic and avoid implementation details unless asked.
""".strip()

EXECUTOR_PROMPT = """
You are an execution agent.
Use the plan to produce the best possible final answer.
If you need assumptions, state them briefly.
""".strip()

print(PLANNER_PROMPT)


## 3) A basic 2-agent LangGraph (Planner → Executor)

LangGraph gives you a small, explicit state machine for agent workflows.

In [ ]:
from __future__ import annotations

from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import START, END, StateGraph

# `add_messages` lets nodes append to `messages` without overwriting history
try:
    from langgraph.graph.message import add_messages
except Exception:  # older versions
    from langgraph.graph import add_messages


class AgentState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    plan: str


def planner_node(state: AgentState) -> AgentState:
    task = state["messages"][-1].content

    plan_msg = llm.invoke([
        SystemMessage(content=PLANNER_PROMPT),
        HumanMessage(content=task),
    ])

    return {
        "plan": plan_msg.content,
        "messages": [AIMessage(content="PLAN:
" + plan_msg.content)],
    }


def executor_node(state: AgentState) -> AgentState:
    task = state["messages"][0].content
    plan = state.get("plan", "")

    final_msg = llm.invoke([
        SystemMessage(content=EXECUTOR_PROMPT),
        HumanMessage(content=f"TASK:
{task}

PLAN:
{plan}"),
    ])

    return {"messages": [final_msg]}


graph = StateGraph(AgentState)
graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", END)

app = graph.compile()
app


## 4) Run it on a task

Change the `task` string and re-run this cell.

In [ ]:
task = """
I have a dataset with customer support tickets.
I want to cluster them into themes and produce a short label + 1-2 sentence summary for each cluster.
Give me a plan and then a suggested approach.
""".strip()

out = app.invoke({"messages": [HumanMessage(content=task)]})

# The executor's final answer is the last message
out["messages"][-1].content


## 5) (Optional) Make it iterative (Executor ⇄ Critic loop)

If you want a slightly more agentic feel, add a third agent that critiques and asks for improvements, then loop a couple of times.

In [ ]:
CRITIC_PROMPT = """
You are a critic agent.
Review the proposed solution for correctness, missing steps, and clarity.
Return (1) a concise critique and (2) concrete improvement requests.
""".strip()

MAX_ITERS = 2

class LoopState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    plan: str
    iter: int


def critic_node(state: LoopState) -> LoopState:
    last_answer = state["messages"][-1].content
    critique = llm.invoke([
        SystemMessage(content=CRITIC_PROMPT),
        HumanMessage(content=last_answer),
    ])
    return {"messages": [AIMessage(content="CRITIQUE:
" + critique.content)]}


def improve_node(state: LoopState) -> LoopState:
    task = state["messages"][0].content
    plan = state.get("plan", "")
    critique = state["messages"][-1].content

    improved = llm.invoke([
        SystemMessage(content=EXECUTOR_PROMPT),
        HumanMessage(content=f"TASK:
{task}

PLAN:
{plan}

{critique}

Produce an improved final answer."),
    ])
    return {"messages": [improved], "iter": state.get("iter", 0) + 1}


def should_continue(state: LoopState) -> str:
    if state.get("iter", 0) >= MAX_ITERS:
        return END
    return "critic"


loop = StateGraph(LoopState)
loop.add_node("planner", planner_node)
loop.add_node("executor", executor_node)
loop.add_node("critic", critic_node)
loop.add_node("improve", improve_node)

loop.add_edge(START, "planner")
loop.add_edge("planner", "executor")
loop.add_edge("executor", "critic")
loop.add_edge("critic", "improve")
loop.add_conditional_edges("improve", should_continue)

loop_app = loop.compile()

out2 = loop_app.invoke({"messages": [HumanMessage(content=task)], "iter": 0})
out2["messages"][-1].content


---

### Notes for workshop participants

- Start by editing `PLANNER_PROMPT` and `EXECUTOR_PROMPT` — the graph stays the same, but behavior changes a lot.
- If you have multiple different roles, add more nodes (e.g., `researcher`, `coder`, `validator`) and route between them.
- Keep temperature low (`temperature=0`) for predictable behavior during debugging.